In [1]:
import gc
from pathlib import Path

import anndata as ad
import pandas as pd
import scanpy as sc
import scranPY as sp
from scipy.sparse import csr_matrix

from lr_loadings_utils import (
    find_threshold_loadings,
    process_loadings,
    extract_ligands_receptors,
    lr_from_micro_to_mono,
    lr_from_mono_to_micro,
    filter_de_genes_micro,
    ranked_genes_df,
    process_de_genes_celltype
)
from adata_processing_utils import (
    scran_norm,
    prepare_scran_de_data_micro_mono,
    load_brain_blood_data,
    filter_and_process_anno_adata,
    create_blood_brain_tf)

2024-07-01 15:44:43.634609: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-07-01 15:44:43.637821: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2024-07-01 15:44:43.689662: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-07-01 15:44:43.689701: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-07-01 15:44:43.691036: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to

In [2]:
project_path = Path(
    f"/sc/arion/projects/psychgen/lbp/data/ScProcesses_brainBlood_nicole/"
)


In [3]:
batch = "batch_1"

In [4]:
adata_blood_brain = create_blood_brain_tf(batch, project_path, write=False, micro_only=False)


In [5]:
adata_blood_brain

AnnData object with n_obs × n_vars = 69986 × 11784
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'predicted_doublets_consensus', 'side', 'pt', 'pt-side', 'tissue', 'chemistry', 'cell_type', 'cell_type.v2', 'cell_type_coarse'
    var: 'feature_types', 'genome', 'mt', 'ribo', 'hb'
    layers: 'counts', 'log1p_norm'

In [17]:
adata_blood_brain.obs[adata_blood_brain.obs['tissue']=="blood"]['cell_type_coarse'].unique()

array(['DC', 'Monocytes', 'B.cells', 'T-cells', 'NKT', 'Platelets',
       'CD16.NK'], dtype=object)

In [18]:
adata_blood_brain.obs[adata_blood_brain.obs['tissue']=="brain"]['cell_type_coarse'].unique()

array(['Microglia', 'Monocytes', 'T-cells', 'CD16.NK', 'Astrocytes',
       'Pericyte'], dtype=object)

In [37]:
adata_blood_brain.obs['cell_type_coarse'].astype(str) + "_" + adata_blood_brain.obs['tissue'].astype(str)

GATTCAGGTCCAACTA_0_0            DC_blood
GTGGGTCTCCAGATCA_0_0            DC_blood
TTGTAGGAGCCCGAAA_0_0            DC_blood
ATTATCCTCCTTTCTC_0_0            DC_blood
ACTTTCAAGGGTGTGT_0_0            DC_blood
                              ...       
TGTTTGTAGGCGAAGG_12_1    Microglia_brain
GATTCGAGTACGGTTT_12_1    Microglia_brain
ACTGATGCATTGACCA_12_1    Microglia_brain
ACGGAAGAGCAGTCTT_12_1    Microglia_brain
TGAGCGCTCAATCAGC_12_1    Microglia_brain
Length: 69986, dtype: object

In [38]:
adata_blood_brain.obs['cell_type.v3'] = adata_blood_brain.obs['cell_type_coarse'].astype(str) + "_" + adata_blood_brain.obs['tissue'].astype(str)


In [39]:
adata_blood_brain.obs['cell_type.v3'].unique()

array(['DC_blood', 'Monocytes_blood', 'B.cells_blood', 'T-cells_blood',
       'NKT_blood', 'Platelets_blood', 'CD16.NK_blood', 'Microglia_brain',
       'Monocytes_brain', 'T-cells_brain', 'CD16.NK_brain',
       'Astrocytes_brain', 'Pericyte_brain'], dtype=object)

In [41]:
sc.write(project_path / batch /"c2c_liana_outputs" / "blood_brain_tf.h5ad", adata_blood_brain)